# Hands-on RAG with Structured Data and Google Gemini

## Project: SAP Support Knowledge Assistant using Excel + JSON

In this hands-on notebook, you will build a complete **Retrieval-Augmented Generation (RAG)** pipeline over structured data.

### You will learn how to:

- Read structured data from **Excel** and **JSON**
- Apply the correct chunking strategy for structured data
- Convert business records into retrieval-friendly text chunks
- Generate embeddings with the **Google GenAI SDK**
- Perform semantic retrieval using cosine similarity
- Send retrieved context to **Google Gemini**
- Build a complete `ask_rag()` function
- Evaluate retrieval quality using **Hit@K**
- Adapt the notebook to your own Excel or JSON files

> **Why this tutorial avoids generic character chunking:**  
> For structured data, splitting every 500 or 1,000 characters can destroy relationships between fields. Here, one meaningful business record becomes one chunk.

## Architecture

```text
Excel rows                         JSON objects
    │                                  │
    ├── Row-wise business chunks       ├── Hierarchy-preserving chunks
    │                                  │
    └───────────────┬──────────────────┘
                    │
               Unified chunks
                    │
            Gemini text embeddings
                    │
              Vector matrix
                    │
User question ──> Query embedding
                    │
             Cosine similarity
                    │
              Top-K retrieval
                    │
          Retrieved structured context
                    │
               Gemini model
                    │
              Grounded answer
```

## Step 1 — Install required libraries

This notebook uses the current **Google GenAI SDK** package: `google-genai`.

We intentionally implement retrieval directly with NumPy so you can clearly understand the RAG flow before introducing LangChain, LlamaIndex, Chroma, FAISS, or SAP HANA Vector Engine.

In [ ]:
!pip install -q -U google-genai pandas openpyxl numpy

## Step 2 — Import libraries

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd

from google import genai
from google.genai import types

pd.set_option("display.max_colwidth", 120)

## Step 3 — Add your Gemini API key securely

### Recommended Colab method

1. Open the **Secrets** panel in Google Colab.
2. Create a secret named `GEMINI_API_KEY`.
3. Paste your Gemini API key.
4. Allow notebook access to the secret.

The cell below first tries Colab Secrets. If the secret is not available, it securely asks for the key at runtime.

In [ ]:
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = None

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass("Enter your Gemini API key: ").strip()

if not GEMINI_API_KEY:
    raise ValueError("A Gemini API key is required.")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client created successfully.")

## Step 4 — Configure the models

This notebook uses:

- `gemini-embedding-001` for text embeddings.
- `gemini-2.5-flash` for response generation.

The embedding model supports retrieval-oriented task types such as `RETRIEVAL_DOCUMENT` and `QUESTION_ANSWERING`.

You can later change the generation model to another Gemini model available to your API project.

In [ ]:
EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-2.5-flash"

EMBEDDING_DIMENSION = 768
TOP_K = 4

print("Embedding model :", EMBEDDING_MODEL)
print("Generation model:", GENERATION_MODEL)

# Part A — Create sample structured data

We will use two structured sources:

### Excel: SAP incident records
Each Excel row represents one complete support incident.

### JSON: SAP knowledge articles
Each JSON object represents one complete knowledge article.

This gives us a realistic multi-source RAG scenario.

## Step 5 — Create sample Excel data

In [ ]:
incident_records = [
    {
        "incident_id": "INC-1001",
        "module": "SAP SD",
        "category": "Sales Order",
        "issue": "Sales order cannot be created because the customer sales area data is missing.",
        "root_cause": "Customer master is not extended to the required sales organization, distribution channel, and division.",
        "resolution": "Extend the customer master to the required sales area and retry sales order creation.",
        "priority": "P2"
    },
    {
        "incident_id": "INC-1002",
        "module": "SAP MM",
        "category": "Purchase Order",
        "issue": "Purchase order creation fails because no valid source of supply is found.",
        "root_cause": "Source list or purchasing info record is missing or invalid for the material and plant.",
        "resolution": "Maintain a valid source list or purchasing info record and repeat purchase order creation.",
        "priority": "P2"
    },
    {
        "incident_id": "INC-1003",
        "module": "SAP FI",
        "category": "Posting",
        "issue": "Accounting document cannot be posted due to an invalid posting period.",
        "root_cause": "The financial posting period is closed for the company code and account type.",
        "resolution": "Open the required posting period in the posting period variant after business approval.",
        "priority": "P1"
    },
    {
        "incident_id": "INC-1004",
        "module": "SAP SD",
        "category": "Pricing",
        "issue": "Sales order has zero price even though a material and customer are present.",
        "root_cause": "A required condition record is missing or the pricing procedure determination is incorrect.",
        "resolution": "Check pricing procedure determination and maintain the required condition records.",
        "priority": "P2"
    },
    {
        "incident_id": "INC-1005",
        "module": "SAP MM",
        "category": "Goods Receipt",
        "issue": "Goods receipt cannot be posted because the purchase order has no remaining open quantity.",
        "root_cause": "The ordered quantity has already been fully received or the delivery-completed indicator is set.",
        "resolution": "Review purchase order history and quantity. Correct the PO only if the business process requires more receipt quantity.",
        "priority": "P3"
    },
    {
        "incident_id": "INC-1006",
        "module": "SAP HCM",
        "category": "Payroll",
        "issue": "Payroll processing stops for an employee due to missing master data.",
        "root_cause": "A mandatory infotype record is missing for the payroll period.",
        "resolution": "Maintain the required employee master-data infotype and rerun payroll.",
        "priority": "P1"
    },
    {
        "incident_id": "INC-1007",
        "module": "SAP BTP",
        "category": "Destination",
        "issue": "A BTP application cannot call the backend system through a destination.",
        "root_cause": "Destination URL, authentication settings, proxy type, or trust configuration is incorrect.",
        "resolution": "Validate destination configuration, connectivity, authentication, and certificate trust.",
        "priority": "P2"
    },
    {
        "incident_id": "INC-1008",
        "module": "SAP Integration Suite",
        "category": "Cloud Integration",
        "issue": "An integration flow fails with HTTP 401 Unauthorized.",
        "root_cause": "The receiver credentials are missing, expired, or incorrectly assigned.",
        "resolution": "Verify the security material, credential alias, authentication type, and receiver authorization.",
        "priority": "P1"
    },
    {
        "incident_id": "INC-1009",
        "module": "SAP ABAP",
        "category": "Runtime Error",
        "issue": "ABAP program terminates with a null reference exception.",
        "root_cause": "An object reference is used before being instantiated.",
        "resolution": "Initialize the object reference before method or attribute access and add defensive checks where appropriate.",
        "priority": "P2"
    },
    {
        "incident_id": "INC-1010",
        "module": "SAP HANA Cloud",
        "category": "SQL",
        "issue": "A SQL query is slow when filtering a very large business table.",
        "root_cause": "The query reads excessive rows, uses inefficient joins, or lacks an appropriate data-access strategy.",
        "resolution": "Review execution plan, filters, joins, data model, partitioning, and workload characteristics.",
        "priority": "P3"
    },
    {
        "incident_id": "INC-1011",
        "module": "SAP AI Core",
        "category": "Deployment",
        "issue": "An AI deployment remains in an error state after model deployment is triggered.",
        "root_cause": "Resource configuration, model artifact, Docker image, credentials, or serving configuration may be invalid.",
        "resolution": "Inspect deployment logs and validate resource plan, artifact path, container image, secrets, and serving configuration.",
        "priority": "P2"
    },
    {
        "incident_id": "INC-1012",
        "module": "SAP Build Process Automation",
        "category": "Workflow",
        "issue": "A workflow waits indefinitely for an approval task.",
        "root_cause": "The approval task has no valid recipient, recipient rule fails, or user assignment is missing.",
        "resolution": "Validate recipient determination, role collection, user assignment, and task configuration.",
        "priority": "P2"
    }
]

incidents_df = pd.DataFrame(incident_records)
excel_path = "sap_incidents.xlsx"
incidents_df.to_excel(excel_path, index=False)

print(f"Created {excel_path} with {len(incidents_df)} rows.")
incidents_df.head()

## Step 6 — Create sample JSON data

In [ ]:
knowledge_data = {
    "knowledge_base": "SAP Support Knowledge",
    "version": "1.0",
    "knowledge_articles": [
        {
            "article_id": "KB-2001",
            "title": "Troubleshooting SAP BTP Destination Connectivity",
            "product": "SAP BTP",
            "tags": ["destination", "connectivity", "authentication", "trust"],
            "symptoms": [
                "Backend call fails",
                "HTTP 401, 403, 502, or timeout",
                "Destination test may fail"
            ],
            "guidance": {
                "checks": [
                    "Verify destination URL and proxy type",
                    "Validate authentication method and credentials",
                    "Check Cloud Connector mapping when OnPremise proxy is used",
                    "Validate certificate trust and role assignments"
                ],
                "expected_outcome": "The application can securely reach the target system."
            }
        },
        {
            "article_id": "KB-2002",
            "title": "Resolving Pricing Problems in SAP Sales Orders",
            "product": "SAP SD",
            "tags": ["pricing", "condition record", "sales order"],
            "symptoms": [
                "Net price is zero",
                "Expected discount is missing",
                "Pricing condition is inactive"
            ],
            "guidance": {
                "checks": [
                    "Check pricing procedure determination",
                    "Check condition records and validity dates",
                    "Check access sequence",
                    "Review pricing analysis in the sales document"
                ],
                "expected_outcome": "The expected pricing conditions are determined correctly."
            }
        },
        {
            "article_id": "KB-2003",
            "title": "Handling HTTP 401 Errors in SAP Integration Suite",
            "product": "SAP Integration Suite",
            "tags": ["cloud integration", "401", "credentials", "security material"],
            "symptoms": [
                "Receiver system rejects request",
                "Integration flow message fails with HTTP 401"
            ],
            "guidance": {
                "checks": [
                    "Confirm receiver authentication type",
                    "Validate credential alias used by the adapter",
                    "Check certificate or OAuth configuration",
                    "Confirm user authorization in the receiver system"
                ],
                "expected_outcome": "The integration flow authenticates successfully with the receiver."
            }
        },
        {
            "article_id": "KB-2004",
            "title": "AI Deployment Troubleshooting in SAP AI Core",
            "product": "SAP AI Core",
            "tags": ["deployment", "serving", "logs", "resource configuration"],
            "symptoms": [
                "Deployment enters ERROR state",
                "Serving endpoint is unavailable"
            ],
            "guidance": {
                "checks": [
                    "Inspect deployment and container logs",
                    "Check model artifact location",
                    "Validate Docker image accessibility",
                    "Validate secrets and resource configuration"
                ],
                "expected_outcome": "The deployment reaches a running state and exposes a serving endpoint."
            }
        },
        {
            "article_id": "KB-2005",
            "title": "Improving SQL Performance in SAP HANA Cloud",
            "product": "SAP HANA Cloud",
            "tags": ["sql", "performance", "execution plan", "filtering"],
            "symptoms": [
                "Query response time is high",
                "Large volume of rows scanned"
            ],
            "guidance": {
                "checks": [
                    "Inspect the execution plan",
                    "Push selective filters as early as possible",
                    "Review join cardinality and data model",
                    "Avoid unnecessary columns and row processing"
                ],
                "expected_outcome": "The query reads less data and executes more efficiently."
            }
        }
    ]
}

json_path = "sap_knowledge.json"
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(knowledge_data, file, indent=2, ensure_ascii=False)

print(f"Created {json_path} with {len(knowledge_data['knowledge_articles'])} knowledge articles.")

# Part B — Load and inspect the structured data

## Step 7 — Read the Excel file

In [ ]:
excel_df = pd.read_excel(excel_path)

print("Excel shape:", excel_df.shape)
excel_df.head(3)

## Step 8 — Read the JSON file

In [ ]:
with open(json_path, "r", encoding="utf-8") as file:
    loaded_json = json.load(file)

print("JSON knowledge base:", loaded_json["knowledge_base"])
print("Number of JSON articles:", len(loaded_json["knowledge_articles"]))

loaded_json["knowledge_articles"][0]

# Part C — Chunk structured data correctly

## Excel strategy: one business row = one chunk

An incident row contains fields that belong together:

- incident ID
- module
- issue
- root cause
- resolution
- priority

Splitting the row in the middle would destroy business meaning.

## JSON strategy: one knowledge article object = one chunk

For nested JSON, we preserve the article hierarchy and flatten the object into a retrieval-friendly text representation.

## Step 9 — Helper function to convert nested JSON into readable text

In [ ]:
def flatten_json_value(value):
    """Convert nested dictionaries/lists/scalars into readable text."""
    if isinstance(value, dict):
        return "; ".join(
            f"{key}: {flatten_json_value(val)}"
            for key, val in value.items()
        )

    if isinstance(value, list):
        return "; ".join(flatten_json_value(item) for item in value)

    return str(value)

## Step 10 — Convert each Excel row into one structured chunk

In [ ]:
def excel_rows_to_chunks(df, source_name="sap_incidents.xlsx"):
    chunks = []

    for row_index, row in df.iterrows():
        clean_record = {
            column: value
            for column, value in row.to_dict().items()
            if pd.notna(value)
        }

        text = "\n".join(
            f"{column}: {value}"
            for column, value in clean_record.items()
        )

        chunks.append({
            "chunk_id": f"excel-{row_index + 1}",
            "text": text,
            "metadata": {
                "source_type": "excel",
                "source_name": source_name,
                "row_number": row_index + 2,
                "record_id": str(clean_record.get("incident_id", "")),
                "module": str(clean_record.get("module", "")),
                "category": str(clean_record.get("category", ""))
            }
        })

    return chunks


excel_chunks = excel_rows_to_chunks(excel_df)

print("Excel chunks:", len(excel_chunks))
print("\nFirst Excel chunk:\n")
print(excel_chunks[0]["text"])
print("\nMetadata:")
print(excel_chunks[0]["metadata"])

## Step 11 — Convert each JSON knowledge article into one chunk

In [ ]:
def json_articles_to_chunks(json_obj, source_name="sap_knowledge.json"):
    chunks = []

    articles = json_obj.get("knowledge_articles", [])

    for index, article in enumerate(articles):
        text = "\n".join(
            f"{key}: {flatten_json_value(value)}"
            for key, value in article.items()
        )

        chunks.append({
            "chunk_id": f"json-{index + 1}",
            "text": text,
            "metadata": {
                "source_type": "json",
                "source_name": source_name,
                "json_path": f"knowledge_articles[{index}]",
                "record_id": str(article.get("article_id", "")),
                "product": str(article.get("product", "")),
                "title": str(article.get("title", ""))
            }
        })

    return chunks


json_chunks = json_articles_to_chunks(loaded_json)

print("JSON chunks:", len(json_chunks))
print("\nFirst JSON chunk:\n")
print(json_chunks[0]["text"])
print("\nMetadata:")
print(json_chunks[0]["metadata"])

## Step 12 — Combine all chunks into one retrieval corpus

In [ ]:
all_chunks = excel_chunks + json_chunks

print("Total chunks:", len(all_chunks))
print("Excel chunks:", len(excel_chunks))
print("JSON chunks :", len(json_chunks))

chunk_preview_df = pd.DataFrame([
    {
        "chunk_id": chunk["chunk_id"],
        "source_type": chunk["metadata"]["source_type"],
        "record_id": chunk["metadata"]["record_id"],
        "preview": chunk["text"][:180] + "..."
    }
    for chunk in all_chunks
])

chunk_preview_df.head(10)

# Part D — Generate embeddings

An embedding converts text into a numerical vector.

Semantically similar texts should have similar vector directions.

For retrieval:

```text
Document chunk ──> document embedding
Question       ──> query embedding
                         │
                  cosine similarity
                         │
                    Top-K chunks
```

We use:

- `RETRIEVAL_DOCUMENT` for stored chunks.
- `QUESTION_ANSWERING` for user questions.

## Step 13 — Embedding helpers

In [ ]:
def normalize_vector(vector):
    vector = np.asarray(vector, dtype=np.float32)
    magnitude = np.linalg.norm(vector)

    if magnitude == 0:
        return vector

    return vector / magnitude


def embed_documents(texts):
    """Embed a list of document chunks in one API request."""
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_DOCUMENT",
            output_dimensionality=EMBEDDING_DIMENSION
        )
    )

    vectors = [
        normalize_vector(embedding.values)
        for embedding in response.embeddings
    ]

    return np.vstack(vectors)


def embed_question(question):
    """Embed one user question for retrieval."""
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=question,
        config=types.EmbedContentConfig(
            task_type="QUESTION_ANSWERING",
            output_dimensionality=EMBEDDING_DIMENSION
        )
    )

    return normalize_vector(response.embeddings[0].values)

## Step 14 — Create the document embedding matrix

This cell calls the Gemini Embeddings API.

The result is:

```text
number of chunks × embedding dimension
```

In [ ]:
document_texts = [chunk["text"] for chunk in all_chunks]

document_embeddings = embed_documents(document_texts)

print("Number of chunks      :", len(all_chunks))
print("Embedding matrix shape:", document_embeddings.shape)

# Part E — Semantic retrieval

Because vectors are normalized, cosine similarity becomes a simple dot product.

Higher score means the query and chunk are more semantically similar.

## Step 15 — Create the retrieval function

In [ ]:
def retrieve(question, top_k=TOP_K):
    query_vector = embed_question(question)

    scores = document_embeddings @ query_vector

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        chunk = all_chunks[int(index)]

        results.append({
            "rank": rank,
            "score": float(scores[index]),
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "metadata": chunk["metadata"]
        })

    return results

## Step 16 — Test retrieval without the LLM

This is an important RAG debugging step.

Always inspect retrieval before blaming the language model.

In [ ]:
test_question = "My SAP BTP application cannot connect to the backend through a destination. What should I check?"

retrieved = retrieve(test_question, top_k=4)

for item in retrieved:
    print("=" * 100)
    print(f"Rank: {item['rank']} | Score: {item['score']:.4f}")
    print("Metadata:", item["metadata"])
    print(item["text"])

# Part F — Generate a grounded answer with Gemini

The model will receive:

1. The user's question.
2. Only the most relevant retrieved chunks.
3. Strict grounding instructions.

The model is instructed to say when the answer is not available in the retrieved context.

## Step 17 — Build the RAG prompt

In [ ]:
def build_rag_prompt(question, retrieved_results):
    context_blocks = []

    for item in retrieved_results:
        metadata = item["metadata"]

        context_blocks.append(
            f"""
SOURCE {item['rank']}
Record ID: {metadata.get('record_id', '')}
Source type: {metadata.get('source_type', '')}
Source name: {metadata.get('source_name', '')}
Similarity score: {item['score']:.4f}

CONTENT:
{item['text']}
""".strip()
        )

    context = "\n\n" + "\n\n".join(context_blocks)

    prompt = f"""
You are an SAP support knowledge assistant.

Answer the user's question using ONLY the retrieved context below.

Rules:
1. Do not invent SAP transactions, configuration steps, root causes, or facts.
2. If the answer is not supported by the context, say:
   "The available structured data does not contain enough information to answer this confidently."
3. Prefer concise, practical answers.
4. When possible, structure the answer as:
   - Likely cause
   - What to check
   - Recommended action
5. Cite supporting record IDs in square brackets, for example [INC-1007] or [KB-2001].

USER QUESTION:
{question}

RETRIEVED CONTEXT:
{context}

FINAL ANSWER:
""".strip()

    return prompt

## Step 18 — Build the complete `ask_rag()` pipeline

In [ ]:
def ask_rag(question, top_k=TOP_K, show_sources=True):
    retrieved_results = retrieve(question, top_k=top_k)

    prompt = build_rag_prompt(
        question=question,
        retrieved_results=retrieved_results
    )

    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt
    )

    answer = response.text

    result = {
        "question": question,
        "answer": answer,
        "retrieved_results": retrieved_results
    }

    if show_sources:
        print("QUESTION")
        print(question)
        print("\nANSWER")
        print(answer)
        print("\nRETRIEVED SOURCES")

        for item in retrieved_results:
            metadata = item["metadata"]
            print(
                f"- Rank {item['rank']} | "
                f"Score {item['score']:.4f} | "
                f"Record {metadata.get('record_id', '')} | "
                f"Source {metadata.get('source_name', '')}"
            )

    return result

## Step 19 — Ask your first RAG question

In [ ]:
result = ask_rag(
    "Our BTP application fails to reach an on-premise backend through a destination. "
    "What are the likely causes and checks?"
)

## Step 20 — Try more questions

In [ ]:
questions = [
    "Why does an SAP Integration Suite flow fail with HTTP 401?",
    "My sales order has zero price. What should I check?",
    "How can I troubleshoot an AI deployment that remains in error state?",
    "Why can an accounting document fail because of a posting period?",
    "How can I improve a slow SQL query in SAP HANA Cloud?"
]

for question in questions:
    print("\n" + "#" * 110)
    ask_rag(question, top_k=3, show_sources=True)

# Part G — Inspect retrieval results in a table

This helper is useful while debugging relevance.

In [ ]:
def retrieval_table(question, top_k=5):
    results = retrieve(question, top_k=top_k)

    return pd.DataFrame([
        {
            "rank": item["rank"],
            "score": round(item["score"], 4),
            "record_id": item["metadata"].get("record_id", ""),
            "source_type": item["metadata"].get("source_type", ""),
            "source_name": item["metadata"].get("source_name", ""),
            "preview": item["text"][:220] + "..."
        }
        for item in results
    ])


retrieval_table(
    "The receiver is rejecting my integration flow because authentication failed.",
    top_k=5
)

# Part H — Evaluate retrieval quality with Hit@K

A RAG system can fail even when the language model is strong.

One of the first evaluation questions should be:

> **Did retrieval return the correct supporting record in the top K results?**

We create a small evaluation set containing:

- a user question
- the expected relevant record ID

## Step 21 — Create evaluation questions

In [ ]:
evaluation_set = [
    {
        "question": "The BTP destination cannot reach my backend system. What should I validate?",
        "expected_record_id": "INC-1007"
    },
    {
        "question": "Why is my sales order price zero?",
        "expected_record_id": "INC-1004"
    },
    {
        "question": "My Cloud Integration flow receives HTTP 401 Unauthorized.",
        "expected_record_id": "INC-1008"
    },
    {
        "question": "The AI Core deployment is in error state.",
        "expected_record_id": "INC-1011"
    },
    {
        "question": "The financial posting period is closed.",
        "expected_record_id": "INC-1003"
    },
    {
        "question": "A HANA query scans too much data and is slow.",
        "expected_record_id": "INC-1010"
    }
]

pd.DataFrame(evaluation_set)

## Step 22 — Calculate Hit@K

In [ ]:
def evaluate_hit_at_k(evaluation_data, k=3):
    evaluation_results = []

    for item in evaluation_data:
        retrieved_results = retrieve(item["question"], top_k=k)

        retrieved_ids = [
            result["metadata"].get("record_id", "")
            for result in retrieved_results
        ]

        hit = item["expected_record_id"] in retrieved_ids

        evaluation_results.append({
            "question": item["question"],
            "expected_record_id": item["expected_record_id"],
            "retrieved_ids": retrieved_ids,
            f"hit@{k}": hit
        })

    results_df = pd.DataFrame(evaluation_results)
    hit_rate = results_df[f"hit@{k}"].mean()

    print(f"Hit@{k}: {hit_rate:.2%}")

    return results_df


evaluation_results_df = evaluate_hit_at_k(evaluation_set, k=3)
evaluation_results_df

# Part I — Use your own Excel or JSON file

The following optional section lets you upload your own structured file.

## Recommended chunking rules

### Excel

Use one row per chunk when one row represents one business entity, such as:

- employee
- sales order
- product
- incident
- invoice
- customer
- course

### JSON

Keep one logical object together when it represents one complete entity.

For deeply nested JSON, preserve important parent-child context in the chunk metadata or text.

> Do not blindly use `RecursiveCharacterTextSplitter` for structured business data.

## Step 23 — Optional file uploader for Colab

In [ ]:
# Run this cell only when you want to upload your own .xlsx or .json file.

# from google.colab import files
# uploaded = files.upload()
# uploaded_filenames = list(uploaded.keys())
# print(uploaded_filenames)

## Step 24 — Generic Excel loader for one-row-per-chunk data

In [ ]:
def load_custom_excel_as_chunks(file_path, sheet_name=0):
    df = pd.read_excel(file_path, sheet_name=sheet_name)

    chunks = []

    for row_index, row in df.iterrows():
        record = {
            column: value
            for column, value in row.to_dict().items()
            if pd.notna(value)
        }

        text = "\n".join(
            f"{column}: {value}"
            for column, value in record.items()
        )

        chunks.append({
            "chunk_id": f"custom-excel-{row_index + 1}",
            "text": text,
            "metadata": {
                "source_type": "excel",
                "source_name": str(file_path),
                "row_number": row_index + 2
            }
        })

    return df, chunks


# Example:
# custom_df, custom_excel_chunks = load_custom_excel_as_chunks("my_data.xlsx")
# print(custom_excel_chunks[0])

## Step 25 — Generic JSON loader for a list of business objects

This helper is suitable when the JSON root is a list of records:

```json
[
  {"id": 1, "name": "A", "category": "X"},
  {"id": 2, "name": "B", "category": "Y"}
]
```

For deeply nested domain-specific JSON, customize the entity extraction logic so each chunk corresponds to a meaningful business object.

In [ ]:
def load_custom_json_list_as_chunks(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    if not isinstance(data, list):
        raise ValueError(
            "This generic helper expects the JSON root to be a list of business objects."
        )

    chunks = []

    for index, record in enumerate(data):
        if not isinstance(record, dict):
            raise ValueError(
                f"Record at index {index} is not a JSON object."
            )

        text = "\n".join(
            f"{key}: {flatten_json_value(value)}"
            for key, value in record.items()
        )

        chunks.append({
            "chunk_id": f"custom-json-{index + 1}",
            "text": text,
            "metadata": {
                "source_type": "json",
                "source_name": str(file_path),
                "json_index": index
            }
        })

    return data, chunks


# Example:
# custom_json_data, custom_json_chunks = load_custom_json_list_as_chunks("my_data.json")
# print(custom_json_chunks[0])

# Part J — Rebuild the vector index after changing data

Whenever you add, delete, or modify chunks, regenerate the embeddings.

Example:

```python
all_chunks = custom_excel_chunks + custom_json_chunks
document_texts = [chunk["text"] for chunk in all_chunks]
document_embeddings = embed_documents(document_texts)
```

Then the existing `retrieve()` and `ask_rag()` functions will use the new corpus because they refer to the global variables `all_chunks` and `document_embeddings`.

# RAG troubleshooting checklist

## Problem 1 — Wrong chunks are retrieved

Check:

- Is each chunk a complete business entity?
- Are important column names preserved in the chunk text?
- Is metadata meaningful?
- Is `TOP_K` too small?
- Are duplicate or low-quality records present?
- Does the user's terminology differ greatly from the source terminology?

## Problem 2 — Correct chunk is retrieved but answer is wrong

Check:

- Does the prompt explicitly require grounding?
- Is the retrieved context clear enough?
- Does the prompt tell the model not to invent facts?
- Are conflicting records present?

## Problem 3 — API quota or rate-limit errors

For a tutorial-sized dataset, keep the number of chunks small. For large datasets:

- batch embedding jobs may be more appropriate,
- persist vectors so you do not re-embed unchanged data on every run,
- use a production vector database such as SAP HANA Cloud Vector Engine, Chroma, Qdrant, Weaviate, or Pinecone depending on your architecture.

# What you have built

You implemented the complete RAG flow:

```text
Structured Excel + JSON
        ↓
Business-aware chunking
        ↓
Unified text chunks + metadata
        ↓
Google Gemini embeddings
        ↓
Vector similarity retrieval
        ↓
Top-K supporting records
        ↓
Grounded Gemini prompt
        ↓
Answer with source record IDs
```

## Key learning

For structured data, the most important decision is usually not:

> "Should chunk size be 500 or 1,000 characters?"

Instead ask:

> **"What is the smallest complete business entity that should remain understandable by itself?"**

Examples:

- one incident
- one employee
- one course
- one customer
- one product
- one sales order
- one knowledge article

That entity should normally become the primary chunk.

# Further extensions

After completing this notebook, try these upgrades:

1. Replace NumPy retrieval with **SAP HANA Cloud Vector Engine**.
2. Add metadata filtering such as `module = SAP SD`.
3. Add hybrid search combining semantic similarity and exact filters.
4. Add reranking.
5. Add conversation memory.
6. Add a Streamlit chat interface.
7. Convert the RAG flow into a LangGraph agent.
8. Evaluate answer faithfulness and context relevance.
9. Add citations that link back to the original Excel row or JSON path.
10. Persist embeddings to disk or a vector database.

---

## Official references

- Google GenAI SDK: https://ai.google.dev/gemini-api/docs/libraries
- Gemini embeddings: https://ai.google.dev/gemini-api/docs/embeddings
- Gemini text generation: https://ai.google.dev/gemini-api/docs/text-generation
- Gemini models: https://ai.google.dev/gemini-api/docs/models